[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/bonelab_inverse_dynamics/blob/main/notebooks/tutorial_two_bricks_2d.ipynb)

# Two bricks on a hinge: the first joint torque, and a residual whose right answer is zero

Chapter 1, page 2 of 2 (known answers you can check by hand).
Previous: [one brick](tutorial_brick_2d.ipynb). Next: [squat-to-stand through the real pipeline](tutorial_s2s_markers.ipynb).

We are going to demonstrate the following key steps in the inverse dynamics workflow:

1. **The floor's load as a wrench about a fixed point**, not a centre of pressure (Step 2). The COP is only ever derived, for looking at.
2. **A joint centre carried as a virtual point** on two marker clusters (Step 5). Both bricks put the hinge in the same place, 2.8e-16 m apart.
3. **The bottom-up recursion gives a joint torque** (Step 7): peak hinge torque 7.30 N m, with the residual at the free top down at 8.3e-11 N and 6.4e-05 N m.
4. **The same hinge torque top-down**, from the upper brick alone (Step 8). The two routes agree to 6.4e-05 N m, and that gap *is* the residual.
5. **Break the model and the residual goes off** (Step 8): make the lower brick 20 % heavier and the residual picks up +7.85 N, exactly the invented weight.
6. **The energy audit closes with three powers** (floor, hinge, residual) (Step 9): peak imbalance 1.66e-04 W against a 13.8 W peak.

The setup: two bricks, hinged. The lower one is bolted to the floor (the job
the foot does in the squat-to-stand on the next page) and the upper one swings
about the hinge. So now there's something measured at the bottom (the floor
pushes back), one unknown in the middle (the hinge wrench, which is all "a
joint torque" is), and nothing touching the top. That last bit is why this
page exists: for the first time the residual has a known right answer, zero.

In [1]:
# Colab bootstrap — a no-op when running locally in the uv environment.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "boneid @ git+https://github.com/jeremydwong/bonelab_inverse_dynamics"],
                   check=True)
os.makedirs("../reports", exist_ok=True)

## Study config — the only thing you edit

In [2]:
import numpy as np

RATE_HZ = 500.0
DURATION_S = 2.0
CONTACT_THRESHOLD_N = 20.0      # vertical force marking floor contact
KINEMATIC_LOWPASS_HZ = 0.0      # analytic data: no filtering
LAB_ORIGIN = np.zeros(3)        # the fixed point every external moment is taken about

## Step 1 — mocap data: four corner markers per brick

Eight markers, four non-coplanar corners on each brick, named `a1..a4`
(lower) and `b1..b4` (upper). The lower brick's four never move. As before
we also receive the simulator's own skeleton and poses, used only for
grading.

In [3]:
from boneid.simulate import simulate_two_bricks_2d

skeleton_true, kin_true, ground, markers, hinge_local = simulate_two_bricks_2d(
    rate=RATE_HZ, duration=DURATION_S)
t = kin_true.t
print(f"{len(t)} frames at {RATE_HZ:.0f} Hz, {len(markers)} markers: {list(markers)}")
print("hinge, measured with a ruler in each brick's own frame:",
      {k: np.round(v, 3).tolist() for k, v in hinge_local.items()})

1000 frames at 500 Hz, 8 markers: ['a1', 'a2', 'a3', 'a4', 'b1', 'b2', 'b3', 'b4']
hinge, measured with a ruler in each brick's own frame: {'lower': [0.0, 0.0, 0.0], 'upper': [0.0, 0.0, -0.45]}


In [4]:
import warnings
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from boneid import report as R
from boneid.report import JOINT_COLORS, FAINT, NEUTRAL, new_fig, new_grid, style_axes
warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")

def show(fig):
    display(HTML(f'<img src="{R.fig_svg(fig)}" style="max-width:100%">'))

fig, ax = new_fig(height=3.6, width=6.0)
for name, xyz in markers.items():
    col = JOINT_COLORS[0] if name[0] == "a" else JOINT_COLORS[1]
    ax.plot(xyz[:, 0], xyz[:, 2], color=col, lw=1.2, alpha=0.9)
    ax.plot(xyz[0, 0], xyz[0, 2], "o", color=col, ms=4)
ax.plot([], [], color=JOINT_COLORS[0], label="lower brick corners")
ax.plot([], [], color=JOINT_COLORS[1], label="upper brick corners")
ax.axhline(0, color=FAINT, lw=1)
ax.set_aspect("equal")
style_axes(ax, "x, forward (m)", "z, up (m)")
ax.legend(frameon=False, fontsize=9)
show(fig)

## Step 2 — force data: the floor's load as a wrench about a fixed point

A force plate gives you a force $F$ and a moment. You can quote the moment
about a fixed point (the plate origin), or turn it into a **centre of
pressure** (COP): the spot on the surface where the force would have to act to
make that moment with no twist. The COP is intuitive, but getting it means
dividing by the vertical force, which is tiny at the start and end of every
contact.

So boneid keeps the (force, moment about a fixed point) form everywhere. I call
it the *wrench about the lab origin*. You lose nothing (`cop_from_wrench`
converts whenever you want to look) and nothing ever blows up. The plot shows
both: the forces, and the COP recovered from the wrench.

Notice the COP leaves the lower brick's footprint (dashed lines) when the swing
gets vigorous. A bolted brick can hold a moment about its base, so that's fine.
A free-standing brick can't (the floor only pushes), and it would have tipped
right there. That's what the COP is good for: telling you whether a contact is
plausible. It's a derived quantity, not an input.

In [5]:
from boneid.core import cop_from_wrench

print("ground wrench: force", ground.force.shape, "| moment about", ground.point, ground.moment.shape)
cop, _ = cop_from_wrench(ground.force, ground.moment, ground.point, plane_height=0.0)
bw = skeleton_true.mass.sum() * 9.81
fig, axes = new_grid(1, 2, height=2.9)
axes[0, 0].plot(t, ground.force[:, 2], color=JOINT_COLORS[0], lw=1.6, label="vertical")
axes[0, 0].plot(t, ground.force[:, 0], color=JOINT_COLORS[1], lw=1.6, label="fore-aft")
axes[0, 0].axhline(bw, color=FAINT, lw=1, ls="--")
axes[0, 0].annotate("weight of both bricks", (t[0], bw), va="bottom", fontsize=8.5, color=NEUTRAL)
style_axes(axes[0, 0], "time (s)", "floor force (N)")
axes[0, 0].legend(frameon=False, fontsize=8.5)
axes[0, 1].plot(t, cop[:, 0], color=JOINT_COLORS[2], lw=1.6)
for x, lab in ((0.075, "front edge"), (-0.075, "back edge")):
    axes[0, 1].axhline(x, color=FAINT, lw=1, ls="--")
    axes[0, 1].annotate(lab, (t[-1], x), ha="right", va="bottom", fontsize=8.5, color=NEUTRAL)
style_axes(axes[0, 1], "time (s)", "centre of pressure, x (m)")
show(fig)

ground wrench: force (1000, 3) | moment about [0. 0. 0.] (1000, 3)


## Step 3 — skeleton: two bricks

Same as before, twice. The chain is ordered from the far end toward the free
end: `[lower, upper]`. Each brick's frame has its origin at its *proximal* end
— the end nearer the top of the chain — so the lower brick's origin is the
hinge and the upper brick's origin is its own top face.

In [6]:
from boneid.core import Skeleton
from boneid.simulate import box_inertia

SIZE = {"lower": np.array([0.15, 0.15, 0.35]), "upper": np.array([0.12, 0.12, 0.45])}
MASS = {"lower": 4.0, "upper": 3.0}
names = ["lower", "upper"]
skeleton = Skeleton(
    segment_names=names, joint_names=["hinge"],
    mass=np.array([MASS[n] for n in names]),
    com_local=np.array([[0, 0, -0.5 * SIZE[n][2]] for n in names]),
    inertia_local=np.stack([box_inertia(MASS[n], SIZE[n]) for n in names]),
    length=np.array([SIZE[n][2] for n in names]))
print(f"modelled mass {skeleton.mass.sum():.1f} kg; joints: {skeleton.joint_names}")

modelled mass 7.0 kg; joints: ['hinge']


## Step 4 — analysis parameters

In [7]:
from boneid.core import AnalysisParams

params = AnalysisParams(contact_threshold_n=CONTACT_THRESHOLD_N,
                        lowpass_hz=KINEMATIC_LOWPASS_HZ, force_lowpass_hz=0.0)
params

AnalysisParams(gravity=array([ 0.  ,  0.  , -9.81]), contact_threshold_n=20.0, lowpass_hz=0.0, force_lowpass_hz=0.0, filter_order=4, treadmill_speed=0.0, flag_crossover=False)

## Step 5 — segment kinematics: the hinge as a virtual point on two clusters

Each cluster gets fitted exactly as on the [one-brick page](tutorial_brick_2d.ipynb).
What's new is the **hinge**. No marker sits on it. Instead I measured its
position in each brick's frame once (the ruler in Step 1), and every frame each
brick carries it along with its fitted pose:

$$\text{hinge}(t) = R_A(t)\,\text{hinge}_A + d_A(t) \;=\; R_B(t)\,\text{hinge}_B + d_B(t).$$

That's the *virtual point* idea, and it's how the Asics running study finds
every joint centre: anatomical markers are worn only for a standing
calibration, their positions get recorded relative to the tracking clusters,
and during running the clusters carry them. The two bricks' hinges should
agree. Here they do to round-off; on real data the gap between them is one
more data-quality number.

In [8]:
from boneid.core import SegmentKinematics, fit_rigid_transform
from boneid.simulate import tetrahedral_corners

pose = {}
for tag, name in (("a", "lower"), ("b", "upper")):
    moving = np.stack([markers[f"{tag}{k + 1}"] for k in range(4)], axis=1)
    pose[name] = fit_rigid_transform(tetrahedral_corners(SIZE[name]), moving)
hinge_from = {n: pose[n][1] + np.einsum("tij,j->ti", pose[n][0], hinge_local[n]) for n in names}
print(f"hinge from lower vs upper cluster: {np.abs(hinge_from['lower'] - hinge_from['upper']).max():.1e} m apart")

hinge = hinge_from["lower"]
top = pose["upper"][1]                                             # upper brick's origin: its top
foot = hinge + np.einsum("tij,j->ti", pose["lower"][0], [0, 0, -SIZE["lower"][2]])
kin = SegmentKinematics(t=t, rate=RATE_HZ,
                        r_world=np.stack([pose["lower"][0], pose["upper"][0]], axis=1),
                        prox_pos=np.stack([hinge, top], axis=1),
                        dist_pos=np.stack([foot, hinge], axis=1))
print(f"fit misfit: {max(pose[n][2].max() for n in names):.1e} m; "
      f"pose error vs script: {np.abs(kin.r_world - kin_true.r_world).max():.1e}")

hinge from lower vs upper cluster: 2.8e-16 m apart
fit misfit: 2.6e-16 m; pose error vs script: 2.7e-15


## Step 6 — contact events

`detect_contact` thresholds the vertical force and returns a mask plus
(onset, offset) index pairs, bridging short dropouts. The bricks never leave
the floor, so there is exactly one bout spanning the trial; in walking this
is the step that finds every stance.

In [9]:
from boneid.core import detect_contact

mask, events = detect_contact(ground.force[:, 2], CONTACT_THRESHOLD_N)
print(f"{len(events)} contact bout(s): frames {events}")

1 contact bout(s): frames [(np.int64(0), np.int64(999))]


## Step 7 — inverse dynamics bottom-up: the hinge torque

One call. The recursion starts at the lower brick with the floor's wrench as
its distal load, solves for the wrench at the hinge, flips its sign (Newton's
third law) and hands it to the upper brick as *its* distal load, then balances
the upper brick. Whatever is left over at the upper brick's free top is the
residual. (I trim 5 frames off each end, where the central differences are
one-sided.)

In [10]:
from boneid.core import inverse_dynamics

idres = inverse_dynamics(skeleton, kin, ground, params)
inner = slice(5, -5)
print(f"peak hinge torque {np.abs(idres.joint_torque[inner, 0, 1]).max():.2f} N m")
print(f"peak residual: force {np.abs(idres.residual_force[inner]).max():.1e} N, "
      f"torque {np.abs(idres.residual_torque[inner]).max():.1e} N m")

peak hinge torque 7.30 N m
peak residual: force 8.3e-11 N, torque 6.4e-05 N m


## Step 8 — the same hinge torque top-down, and the residual as an alarm

The hinge torque is what it takes to swing a 3 kg, 0.45 m brick about its
bottom end. The residual is zero to finite-difference precision, which says
everything below the free top was right: the floor's wrench, two masses, two
geometries, and the timing between force and motion.

**Top-down.** There's a second way to get the hinge: ignore the floor
entirely and balance the upper brick alone, since it touches only the hinge.
Turn it upside down so the hinge is its proximal end, give it a zero distal
load, and its "residual" *is* the hinge wrench (sign flipped, since it's now
the load on the upper brick). The two routes agree, and the bottom-up
residual is nothing but the difference between them, accumulated over the
whole chain. That's what the residual means on real data, I think the most
useful way to read it: the sum of every mistake below the top.

In [11]:
from boneid.core import GroundWrench

L_b = SIZE["upper"][2]
upper_flipped = Skeleton(segment_names=["upper"], joint_names=[], mass=skeleton.mass[1:],
                         com_local=np.array([[0, 0, 0.5 * L_b]]),       # COM now ABOVE the origin
                         inertia_local=skeleton.inertia_local[1:], length=skeleton.length[1:])
kin_flipped = SegmentKinematics(t=t, rate=RATE_HZ, r_world=kin.r_world[:, 1:],
                                prox_pos=kin.dist_pos[:, 1:], dist_pos=kin.prox_pos[:, 1:])
nothing = GroundWrench(t=t, rate=RATE_HZ, force=np.zeros((len(t), 3)), moment=np.zeros((len(t), 3)))
top_down = inverse_dynamics(upper_flipped, kin_flipped, nothing, params)

fig, axes = new_grid(1, 2, height=3.0)
axes[0, 0].plot(t, -top_down.residual_torque[:, 1], color=NEUTRAL, lw=2.6, alpha=0.4, label="top-down (upper brick alone)")
axes[0, 0].plot(t, idres.joint_torque[:, 0, 1], color=JOINT_COLORS[1], lw=1.2, label="bottom-up (through the floor)")
style_axes(axes[0, 0], "time (s)", "hinge torque on the lower brick, y (N m)")
axes[0, 0].legend(frameon=False, fontsize=8.5)
for c, lab in enumerate("xyz"):
    axes[0, 1].plot(t[inner], idres.residual_torque[inner, c], color=JOINT_COLORS[c], lw=1.1, label=f"M{lab}")
style_axes(axes[0, 1], "time (s)", "residual torque at the free top (N m)")
axes[0, 1].legend(frameon=False, fontsize=8.5)
show(fig)
print(f"bottom-up vs top-down hinge torque: {np.abs(idres.joint_torque[inner, 0] + top_down.residual_torque[inner]).max():.1e} N m")

bottom-up vs top-down hinge torque: 6.4e-05 N m


### Break the model: make the lower brick 20 % too heavy

Say I typed the lower brick in as 20 % heavier than it is. The bottom-up hinge
torque shifts, and the residual force at the free top jumps from zero to an
upward force equal to the invented weight (as if someone had to hold the extra
mass up from above). The top-down estimate doesn't move, because it never
used the lower brick. Nothing about the motion changed; the model was just
wrong below the top, and the residual said so with no reference answer to
compare against.

In [12]:
from dataclasses import replace

heavy = replace(skeleton, mass=skeleton.mass * np.array([1.2, 1.0]))
wrong = inverse_dynamics(heavy, kin, ground, params)
fig, axes = new_grid(1, 2, height=2.9)
axes[0, 0].plot(t, idres.joint_torque[:, 0, 1], color=JOINT_COLORS[1], lw=1.6, label="correct masses")
axes[0, 0].plot(t, wrong.joint_torque[:, 0, 1], color=JOINT_COLORS[3], lw=1.6, ls="--", label="lower brick +20 %")
style_axes(axes[0, 0], "time (s)", "hinge torque, y (N m)")
axes[0, 0].legend(frameon=False, fontsize=8.5)
axes[0, 1].plot(t, idres.residual_force[:, 2], color=JOINT_COLORS[1], lw=1.6, label="correct masses")
axes[0, 1].plot(t, wrong.residual_force[:, 2], color=JOINT_COLORS[3], lw=1.6, ls="--", label="lower brick +20 %")
axes[0, 1].axhline(0.2 * MASS["lower"] * 9.81, color=FAINT, lw=1, ls=":")
axes[0, 1].annotate("the invented weight, held from the top", (t[0], 0.2 * MASS["lower"] * 9.81), va="bottom", fontsize=8.5, color=NEUTRAL)
style_axes(axes[0, 1], "time (s)", "residual vertical force (N)")
axes[0, 1].legend(frameon=False, fontsize=8.5)
show(fig)
print(f"mean residual Fz with the wrong mass: {wrong.residual_force[inner, 2].mean():+.2f} N "
      f"(invented weight {0.2 * MASS['lower'] * 9.81:.2f} N, which the free top must now hold up)")

mean residual Fz with the wrong mass: +7.85 N (invented weight 7.85 N, which the free top must now hold up)


## Step 9 — the energy audit: floor, hinge and residual power

Now three powers have to add up. The floor's is zero (a bolted base doesn't
move, so the floor does no work however hard it pushes). The hinge's is torque
times relative angular velocity, plus a force term that vanishes when both
bricks agree on where the hinge is. And then the residual's. Their sum must be
the rate of change of total mechanical energy. It is =)

In [13]:
from boneid.core import energy_audit

audit = energy_audit(skeleton, kin, ground, idres, params)
fig, axes = new_grid(1, 2, height=2.9)
axes[0, 0].plot(t, audit.de_dt, color=NEUTRAL, lw=2.6, alpha=0.45, label="d(KE+PE)/dt")
axes[0, 0].plot(t, audit.joint_power[:, 0], color=JOINT_COLORS[1], lw=1.2, label="hinge power")
axes[0, 0].plot(t, audit.ground_power, color=JOINT_COLORS[0], lw=1.2, label="floor power")
style_axes(axes[0, 0], "time (s)", "power (W)")
axes[0, 0].legend(frameon=False, fontsize=8.5)
axes[0, 1].plot(t[inner], audit.imbalance[inner], color=JOINT_COLORS[2], lw=1.2)
style_axes(axes[0, 1], "time (s)", "imbalance (W)")
show(fig)
print(f"peak imbalance {np.abs(audit.imbalance[inner]).max():.2e} W of {np.abs(audit.de_dt).max():.1f} W peak; "
      f"floor power max {np.abs(audit.ground_power).max():.1e} W")

peak imbalance 1.66e-04 W of 13.8 W peak; floor power max 0.0e+00 W


## Step 10 — the movement in 3-D

Both bricks, the eight markers, and the floor-force arrow drawn where its line
of action meets the floor (the COP, recovered for display only). Watch it step
outside the lower brick at the ends of the swing.

In [14]:
from html import escape
from pathlib import Path
from boneid.report import simulated_viewer_html
from boneid.viz import box_mesh

viewer_html = simulated_viewer_html(skeleton, kin, ground=ground, markers=markers,
                                    segment_meshes={0: box_mesh(SIZE["lower"]), 1: box_mesh(SIZE["upper"])},
                                    decimate=10, camera_offset=(1.3, -1.6, 0.4))
Path("../reports/tutorial_two_bricks_2d_viewer.html").write_text(viewer_html)
display(HTML(f'<iframe srcdoc="{escape(viewer_html, quote=True)}" width="100%" height="420"></iframe>'))

## What to carry to the next page

- **A joint torque is a hinge wrench.** You find it by balancing the segment
  below (bottom-up, through the measured ground load) or the segment above
  (top-down, when nothing else touches it). Legs get done bottom-up because the
  floor is what we can measure.
- **The residual is the disagreement between those two routes**, summed over
  the chain. Zero here because the model was exact. On a person it carries
  soft-tissue motion, joint-centre error and force–motion misalignment, and
  reading it is what the P1 pages are about.
- **Joint centres ride clusters as virtual points.** The fit misfit and the gap
  between two clusters' estimates of one joint are your quality signals.

Next: [squat-to-stand](tutorial_s2s_markers.ipynb), an idealized person
written out as c3d files and run through the real pipeline, on a body where
every number is known.